# Регрессия SI

Задача: предсказать индекс селективности SI = CC50 / IC50 по молекулярным дескрипторам.

SI показывает, во сколько раз эффективная концентрация ниже токсичной. Чем выше SI, тем безопаснее и эффективнее вещество. Значения SI > 8 считаются практически значимыми.

## 1. Импорты

In [1]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Загрузка и подготовка данных

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")

TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]
TARGET = "SI"

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X = data[feature_columns]
y = data[TARGET]

print(f"Объектов: {len(X)}, признаков: {X.shape[1]}")
print(f"SI: min={y.min():.4f}, median={y.median():.2f}, max={y.max():.2f}")
print(f"Пропусков в признаках: {X.isna().sum().sum()}")

Объектов: 969, признаков: 210
SI: min=0.0115, median=3.90, max=15620.60
Пропусков в признаках: 36


## 3. Разбиение и логарифмирование

Делим данные на обучение и тест с помощью группового разбиения. Объекты с одинаковыми значениями молекулярных дескрипторов относим к одной группе, чтобы они не могли одновременно попасть в обучение и тест.

Для подбора гиперпараметров используем групповую кросс-валидацию. Целевую переменную преобразуем с помощью `log1p`, так как распределение SI сильно скошено вправо.

In [3]:
groups = pd.util.hash_pandas_object(X, index=False)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
cv = GroupKFold(n_splits=5)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Обучение: {len(X_train)} объектов, тест: {len(X_test)} объектов")

Обучение: 759 объектов, тест: 210 объектов


## 4. Метрики

MAE и RMSE на исходной шкале (не в логарифмах). При подборе гиперпараметров — MAE на лог-шкале.

### Базовая модель

Сначала проверим простейшую модель, которая для всех объектов предсказывает медианное значение SI в обучающей выборке.

In [4]:
dummy_model = DummyRegressor(strategy="median")
dummy_model.fit(X_train, y_train_log)

dummy_pred_log = dummy_model.predict(X_test)
dummy_pred = np.expm1(dummy_pred_log)

dummy_mae = mean_absolute_error(y_test, dummy_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))

print(
    f"Baseline | Test MAE: {dummy_mae:.2f} | "
    f"Test RMSE: {dummy_rmse:.2f}"
)

Baseline | Test MAE: 136.00 | Test RMSE: 1127.11


## 5. Ridge

In [5]:
ridge_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(random_state=42),
)

ridge_params = {"ridge__alpha": [0.1, 1, 10, 100, 200, 500, 1000]}

ridge_search = GridSearchCV(
    ridge_pipe, ridge_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
ridge_search.fit(X_train, y_train_log, groups=groups_train)

ridge_cv_results = pd.DataFrame({
    "alpha": ridge_params["ridge__alpha"],
    "CV MAE (лог-шкала)": -ridge_search.cv_results_["mean_test_score"],
})
display(ridge_cv_results)

ridge_best = ridge_search.best_estimator_
print(f"Лучший alpha: {ridge_search.best_params_['ridge__alpha']}")
print(f"CV MAE (лог-шкала): {-ridge_search.best_score_:.4f}")

,alpha,CV MAE (лог-шкала)
0,0.1,1.062792
1,1.0,1.035638
2,10.0,1.011073
3,100.0,0.990753
4,200.0,0.988735
5,500.0,1.000451
6,1000.0,1.013026


Лучший alpha: 200
CV MAE (лог-шкала): 0.9887


Посмотрим на тесте.

In [6]:
y_pred_log = ridge_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

ridge_mae = mean_absolute_error(y_true, y_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Ridge | Test MAE: {ridge_mae:.2f} | Test RMSE: {ridge_rmse:.2f}")

Ridge | Test MAE: 138.43 | Test RMSE: 1126.52


MAE модели составляет 138.43, RMSE — 1126.52. Большая разница между метриками показывает, что на отдельных объектах с высоким SI модель допускает очень крупные ошибки.

## 6. Lasso

In [7]:
lasso_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Lasso(random_state=42, max_iter=20000),
)

lasso_params = {"lasso__alpha": [0.001, 0.01, 0.1, 1]}

lasso_search = GridSearchCV(
    lasso_pipe, lasso_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
lasso_search.fit(X_train, y_train_log, groups=groups_train)

lasso_cv_results = pd.DataFrame({
    "alpha": lasso_params["lasso__alpha"],
    "CV MAE (лог-шкала)": -lasso_search.cv_results_["mean_test_score"],
})
display(lasso_cv_results)

lasso_best = lasso_search.best_estimator_
n_total = X.shape[1]
n_nonzero = (lasso_best.named_steps["lasso"].coef_ != 0).sum()
print(f"Лучший alpha: {lasso_search.best_params_['lasso__alpha']}")
print(f"Занулено признаков: {n_total - n_nonzero} из {n_total}")
print(f"CV MAE (лог-шкала): {-lasso_search.best_score_:.4f}")

,alpha,CV MAE (лог-шкала)
0,0.001,1.042607
1,0.010,1.012504
2,0.100,1.025194
3,1.000,1.091502


Лучший alpha: 0.01
Занулено признаков: 118 из 210
CV MAE (лог-шкала): 1.0125


Посмотрим на тесте.

In [8]:
y_pred_log = lasso_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

lasso_mae = mean_absolute_error(y_true, y_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Lasso | Test MAE: {lasso_mae:.2f} | Test RMSE: {lasso_rmse:.2f}")

Lasso | Test MAE: 238.64 | Test RMSE: 1570.22


MAE модели составляет 238.64 — это значительно хуже результата Ridge. Lasso занулил 118 коэффициентов из 210, однако сильное сокращение числа признаков в данном случае привело к ухудшению качества.

## 7. K ближайших соседей

In [9]:
knn_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    KNeighborsRegressor(),
)

knn_params = {
    "kneighborsregressor__n_neighbors": [3, 5, 7, 10, 15, 20],
    "kneighborsregressor__weights": ["uniform", "distance"],
}

knn_search = GridSearchCV(
    knn_pipe, knn_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
knn_search.fit(X_train, y_train_log, groups=groups_train)

knn_cv_results = pd.DataFrame({
    "n_neighbors": knn_search.cv_results_["param_kneighborsregressor__n_neighbors"],
    "weights": knn_search.cv_results_["param_kneighborsregressor__weights"],
    "CV MAE (лог-шкала)": -knn_search.cv_results_["mean_test_score"],
})
display(knn_cv_results.sort_values("CV MAE (лог-шкала)"))

knn_best = knn_search.best_estimator_
print(f"Лучшие параметры: {knn_search.best_params_}")
print(f"CV MAE (лог-шкала): {-knn_search.best_score_:.4f}")

,n_neighbors,weights,CV MAE (лог-шкала)
3,5,distance,0.927777
5,7,distance,0.931005
2,5,uniform,0.940827
7,10,distance,0.942317
4,7,uniform,0.951304
1,3,distance,0.959264
9,15,distance,0.959323
0,3,uniform,0.962703
6,10,uniform,0.969256
11,20,distance,0.977641


Лучшие параметры: {'kneighborsregressor__n_neighbors': 5, 'kneighborsregressor__weights': 'distance'}
CV MAE (лог-шкала): 0.9278


Посмотрим на тесте.

In [10]:
y_pred_log = knn_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

knn_mae = mean_absolute_error(y_true, y_pred)
knn_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"KNN | Test MAE: {knn_mae:.2f} | Test RMSE: {knn_rmse:.2f}")

KNN | Test MAE: 141.59 | Test RMSE: 1105.20


MAE модели составляет 141.59. Результат немного хуже Ridge, но значительно лучше Lasso.

## 8. Случайный лес

In [11]:
rf_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(random_state=42),
)

rf_params = {
    "randomforestregressor__n_estimators": [100, 200, 300],
    "randomforestregressor__max_depth": [10, 20, 30, None],
    "randomforestregressor__min_samples_leaf": [1, 3, 5],
}

rf_search = RandomizedSearchCV(
    rf_pipe, rf_params,
    n_iter=20, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1, random_state=42,
)
rf_search.fit(X_train, y_train_log, groups=groups_train)

rf_cv_results = pd.DataFrame({
    "n_estimators": rf_search.cv_results_["param_randomforestregressor__n_estimators"],
    "max_depth": rf_search.cv_results_["param_randomforestregressor__max_depth"].astype(str),
    "min_samples_leaf": rf_search.cv_results_["param_randomforestregressor__min_samples_leaf"],
    "CV MAE (лог-шкала)": -rf_search.cv_results_["mean_test_score"],
})
display(rf_cv_results.sort_values("CV MAE (лог-шкала)").head(10))

rf_best = rf_search.best_estimator_
print(f"Лучшие параметры: {rf_search.best_params_}")
print(f"CV MAE (лог-шкала): {-rf_search.best_score_:.4f}")

,n_estimators,max_depth,min_samples_leaf,CV MAE (лог-шкала)
18,300,20,1,0.903582
14,300,None,1,0.903881
16,200,30,1,0.904404
1,200,20,3,0.904582
5,200,None,3,0.904806
19,200,10,1,0.905215
4,200,20,5,0.905253
11,200,None,5,0.905265
13,200,10,3,0.905912
17,300,10,3,0.905927


Лучшие параметры: {'randomforestregressor__n_estimators': 300, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__max_depth': 20}
CV MAE (лог-шкала): 0.9036


Посмотрим на тесте.

In [12]:
y_pred_log = rf_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

rf_mae = mean_absolute_error(y_true, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Случайный лес | Test MAE: {rf_mae:.2f} | Test RMSE: {rf_rmse:.2f}")

Случайный лес | Test MAE: 132.05 | Test RMSE: 1123.50


MAE случайного леса составляет 132.05 — это лучший результат на тестовой выборке среди рассмотренных моделей.

## 9. Градиентный бустинг

In [13]:
gb_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    GradientBoostingRegressor(random_state=42),
)

gb_params = {
    "gradientboostingregressor__n_estimators": [100, 200, 300],
    "gradientboostingregressor__max_depth": [3, 5, 7],
    "gradientboostingregressor__learning_rate": [0.01, 0.05, 0.1],
}

gb_search = RandomizedSearchCV(
    gb_pipe, gb_params,
    n_iter=20, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1, random_state=42,
)
gb_search.fit(X_train, y_train_log, groups=groups_train)

gb_cv_results = pd.DataFrame({
    "n_estimators": gb_search.cv_results_["param_gradientboostingregressor__n_estimators"],
    "max_depth": gb_search.cv_results_["param_gradientboostingregressor__max_depth"],
    "learning_rate": gb_search.cv_results_["param_gradientboostingregressor__learning_rate"],
    "CV MAE (лог-шкала)": -gb_search.cv_results_["mean_test_score"],
})
display(gb_cv_results.sort_values("CV MAE (лог-шкала)").head(10))

gb_best = gb_search.best_estimator_
print(f"Лучшие параметры: {gb_search.best_params_}")
print(f"CV MAE (лог-шкала): {-gb_search.best_score_:.4f}")

,n_estimators,max_depth,learning_rate,CV MAE (лог-шкала)
1,200,5,0.05,0.900274
8,100,5,0.05,0.904387
5,300,3,0.05,0.908873
3,100,5,0.10,0.914132
19,100,3,0.10,0.916515
12,300,5,0.01,0.917460
2,100,3,0.05,0.920256
18,300,5,0.10,0.920377
15,200,5,0.10,0.920834
14,100,7,0.05,0.932473


Лучшие параметры: {'gradientboostingregressor__n_estimators': 200, 'gradientboostingregressor__max_depth': 5, 'gradientboostingregressor__learning_rate': 0.05}
CV MAE (лог-шкала): 0.9003


Посмотрим на тесте.

In [14]:
y_pred_log = gb_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

gb_mae = mean_absolute_error(y_true, y_pred)
gb_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Градиентный бустинг | Test MAE: {gb_mae:.2f} | Test RMSE: {gb_rmse:.2f}")

Градиентный бустинг | Test MAE: 132.67 | Test RMSE: 1122.80


MAE градиентного бустинга составляет 132.67. Результат практически совпадает со случайным лесом: разница по MAE составляет только 0.62.

## 10. Сравнение

In [15]:
results = pd.DataFrame({
    "Модель": [
        "Baseline",
        "Ridge",
        "Lasso",
        "KNN",
        "Случайный лес",
        "Градиентный бустинг",
    ],
    "MAE": [
        dummy_mae,
        ridge_mae,
        lasso_mae,
        knn_mae,
        rf_mae,
        gb_mae,
    ],
    "RMSE": [
        dummy_rmse,
        ridge_rmse,
        lasso_rmse,
        knn_rmse,
        rf_rmse,
        gb_rmse,
    ],
})

results = results.sort_values("MAE").reset_index(drop=True)
display(results.round(2))

,Модель,MAE,RMSE
0,Случайный лес,132.05,1123.50
1,Градиентный бустинг,132.67,1122.80
2,Baseline,136.00,1127.11
3,Ridge,138.43,1126.52
4,KNN,141.59,1105.20
5,Lasso,238.64,1570.22


Случайный лес и градиентный бустинг показали лучшие результаты: MAE 132.05 и 132.67 соответственно. Однако baseline получил MAE 136.00, поэтому улучшение составляет всего около 3%.

Ridge и KNN с MAE 138.43 и 141.59 не превзошли базовую модель. Худший результат показал Lasso с MAE 238.64.

Таким образом, модели слабо улучшают прогноз по сравнению с простым предсказанием медианы. Молекулярные дескрипторы в текущем виде недостаточно хорошо объясняют значения SI на исходной шкале.

У всех моделей RMSE значительно выше MAE. Это связано с отдельными объектами с экстремально высокими значениями SI, на которых возникают крупные ошибки.

Для улучшения результата можно дополнительно исследовать выбросы, проверить другие преобразования целевой переменной и использовать дополнительные химические признаки.

Групповое разбиение исключает попадание объектов с одинаковыми дескрипторами одновременно в обучение и тест.